# Introduction

This is a consistency check to see if trip generation and trip distribution/mode choice are consistent.

In [1]:
import pandas as pd 
import caliperpy
import os
import matplotlib.pyplot as plt
import numpy as np

def read_bin(filename):
    dk = caliperpy.TransCAD.connect()
    out = None
    try:
        out = dk.GetDataFrameFromBin(filename)
    except Exception as e:
        print(e) # type: ignore
    finally:
        dk.Close()
        caliperpy.TransCAD.disconnect()
    return out

def read_mtx(filename, dk):
    # Reads a matrix
    # dk = caliperpy.TransCAD.connect()
    out_dict = {}
    try:
        mtx = dk.OpenMatrix(filename, "True")
        cores = dk.GetMatrixCoreNames(mtx)
        for c in cores: # type: ignore
            m1 = dk.CreateMatrixCurrency(mtx, c, None, None, None)
            out_dict[c] = np.nan_to_num(np.array(dk.GetMatrixValues(m1, None, None), dtype=np.float32))
    except Exception as e:
        print(e) # type: ignore
    # finally:
        # dk.Close()
        # caliperpy.TransCAD.disconnect()
    return out_dict

def read_length_mtx(filename, dk):
    # Reads a matrix
    # dk = caliperpy.TransCAD.connect()
    out_dict = {}
    try:
        mtx = dk.OpenMatrix(filename, "True")
        # cores = dk.GetMatrixCoreNames(mtx)
        # for c in cores: # type: ignore
        #     m1 = dk.CreateMatrixCurrency(mtx, c, None, None, None)
        #     out_dict[c] = np.nan_to_num(np.array(dk.GetMatrixValues(m1, None, None), dtype=np.float32))
        m1 = dk.CreateMatrixCurrency(mtx, "Length (Skim)", None, None, None)    
        out = np.nan_to_num(np.array(dk.GetMatrixValues(m1, None, None), dtype=np.float32))
    except Exception as e:
        print(e) # type: ignore
    # finally:
        # dk.Close()
        # caliperpy.TransCAD.disconnect()
    return out

SCENARIO_BASE_FOLDER = r'C:\models\Reno_TDM\scenarios'
SCENARIO_CONTROL = 'base_2022'
SCENARIO_CHANGE = 'base_2022_resnets_dl_s'
PERIODS = ['AM', 'MD', 'PM', 'NT']

purp_list = {
    'W_HBW': ['v0', 'ilvi', 'ilvs', 'ihvi', 'ihvs'],
    'W_HBO': ['v0', 'vi', 'vs'],
    'N_HBSCH': ['v0', 'vi', 'vs'],
    'N_HBSHP': ['v0', 'vi', 'vs'],
    'N_HBSR': ['v0', 'vi', 'vs'],
    'N_HBO': ['v0', 'vi', 'vs']
}

field_list = []

for k in purp_list.keys():
    for v in purp_list[k]:
        field_list.append(f"{k}_{v}")
        
sedata_base = read_bin(os.path.join(SCENARIO_BASE_FOLDER, SCENARIO_CONTROL, 'output', 'sedata', 'scenario_se.bin'))
sedata_chng = read_bin(os.path.join(SCENARIO_BASE_FOLDER, SCENARIO_CHANGE, 'output', 'sedata', 'scenario_se.bin'))

dk = caliperpy.TransCAD.connect()

base_totals = {}
chng_totals = {}
skim = {}

for purp in purp_list.keys():
    print(f"Working on purpose {purp}")
    for per in PERIODS:
        input = read_mtx(os.path.join(SCENARIO_BASE_FOLDER, SCENARIO_CONTROL, 'output', 'resident', 'trip_matrices', f'pa_per_trips_{purp}_{per}.mtx'), dk)
        for vs in purp_list[purp]:
            if not (purp, per, vs) in base_totals:
                base_totals[(purp, per, vs)] = input[f'dc_{vs}'].sum()
            else:
                base_totals[(purp, per, vs)] += input[f'dc_{vs}'].sum()

        input = read_mtx(os.path.join(SCENARIO_BASE_FOLDER, SCENARIO_CHANGE, 'output', 'resident', 'trip_matrices', f'pa_per_trips_{purp}_{per}.mtx'), dk)
        for vs in purp_list[purp]:
            if not (purp, per, vs) in chng_totals:
                chng_totals[(purp, per, vs)] = input[f'dc_{vs}'].sum()
            else:
                chng_totals[(purp, per, vs)] += input[f'dc_{vs}'].sum()

caliperpy.TransCAD.disconnect()

Connecting to TransCAD...


c:\ProgramData\anaconda3\envs\fhwa_ai2\Lib\site-packages\caliperpy\caliper3.py:1073: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].mask(df[col] == '', None, inplace=True)
c:\ProgramData\anaconda3\envs\fhwa_ai2\Lib\site-packages\caliperpy\caliper3.py:1073: ChainedAssignmentError: A value is being set on a copy of a DataFrame o

Disconnected from TransCAD!
Connecting to TransCAD...
Disconnected from TransCAD!
Connecting to TransCAD...
Working on purpose W_HBW


c:\ProgramData\anaconda3\envs\fhwa_ai2\Lib\site-packages\caliperpy\caliper3.py:1073: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].mask(df[col] == '', None, inplace=True)
c:\ProgramData\anaconda3\envs\fhwa_ai2\Lib\site-packages\caliperpy\caliper3.py:1073: ChainedAssignmentError: A value is being set on a copy of a DataFrame o

Working on purpose W_HBO
Working on purpose N_HBSCH
Working on purpose N_HBSHP
Working on purpose N_HBSR
Working on purpose N_HBO
Disconnected from TransCAD!


True

In [2]:
purp_tod_fields = []
for purp in purp_list.keys():
    for vs in purp_list[purp]:
        for per in PERIODS:
            purp_tod_fields.append(f'{purp}_{vs}_{per}')
gen_data = pd.concat([sedata_base[purp_tod_fields].sum(), sedata_chng[purp_tod_fields].sum()], axis = 1).rename(columns = {0: 'base_gen', 1: 'chng_gen'}).reset_index()
# gen_data.style.format('{:,.1f}')

In [3]:
gen_data['base_dist'] = gen_data['index'].apply(lambda x: base_totals[(f'{x.split("_")[0]}_{x.split("_")[1]}', f'{x.split("_")[3]}', f'{x.split("_")[2]}')])
gen_data['chng_dist'] = gen_data['index'].apply(lambda x: chng_totals[(f'{x.split("_")[0]}_{x.split("_")[1]}', f'{x.split("_")[3]}', f'{x.split("_")[2]}')])

gen_data.loc[44:].style.format({'base_gen': '{:,.1f}', 'chng_gen': '{:,.1f}', 'base_dist': '{:,.1f}', 'chng_dist': '{:,.1f}'})

,index,base_gen,chng_gen,base_dist,chng_dist
44,N_HBSHP_v0_AM,"1,423.5","1,423.5","1,423.3","1,423.3"
45,N_HBSHP_v0_MD,"11,305.6","11,305.6","11,304.7","11,304.7"
46,N_HBSHP_v0_PM,"5,936.4","5,936.4","5,936.0","5,936.0"
47,N_HBSHP_v0_NT,"4,348.2","4,348.2","4,347.9","4,347.9"
48,N_HBSHP_vi_AM,"6,755.0","6,755.0","6,754.7","6,754.7"
49,N_HBSHP_vi_MD,"45,969.8","45,969.8","45,967.9","45,967.9"
50,N_HBSHP_vi_PM,"27,266.3","27,266.3","27,265.6","27,265.6"
51,N_HBSHP_vi_NT,"30,009.9","30,009.9","30,009.3","30,009.3"
52,N_HBSHP_vs_AM,"4,205.1","4,205.1","4,205.0","4,205.0"
53,N_HBSHP_vs_MD,"30,343.2","30,343.2","30,342.8","30,342.8"
